In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Neural network example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [2]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

# Преглед на данните
df.show(5)

+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|Gender|Age|   Degree|PlatformUsage|PlatformHours|OutPlatformHours|Satisfaction|Self-assessment|PreferredFormat|
+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|     Ж| 21|Бакалавър|           Да|            6|               2|           4|              4|          Видео|
|     М| 23| Магистър|           Да|            8|               1|           5|              5|          Видео|
|     Ж| 20|Бакалавър|           Не|            0|               3|           3|              3|          Текст|
|     Ж| 22|Бакалавър|           Да|            5|               1|           4|              4|         Смесен|
|     М| 24| Магистър|           Да|            9|               0|           5|              5|          Видео|
+------+---+---------+-------------+-------------+----------------+------------+---------------+

In [12]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Подготовка на данните
feature_cols = ["Age","PlatformHours","OutPlatformHours","Self-assessment"]
# Обединяване на всички характеристики във features колона
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df)

# Разделяне на данните
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

# Архитектура на мрежата
layers = [4, 7, 8, 6]
# Създаване на модела
mlp = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="Satisfaction",
    layers=layers, maxIter=100)
# Обучение
model = mlp.fit(train_data)

# Прогнозиране
predictions = model.transform(test_data)
# Оценка
evaluator = MulticlassClassificationEvaluator(
    labelCol="Satisfaction", predictionCol="prediction",
    metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 0.75
